# Combine Seed Plots (List Based)

This notebook combines PNG images from different seed folders vertically. It uses an explicit list of experiments instead of auto-searching prefix patterns.

In [1]:
import os
import glob
from PIL import Image

seeds_to_merge = [-1, 0, 1, 2, 3, 7]
experiments_to_plot = [
    "query_mask_5_Seed", 
    "query_mask_15_fake_seed"
]

# --- Paths ---
INPUT_BASE_DIR = r"C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\TM_Score"
OUTPUT_DIR = r"C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds"


# Process and Combine Images

In [ ]:
# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

for exp in experiments_to_plot:
    print(f"\n--- Processing experiment: {exp} ---")
    
    # 1. Create the list of folder names
    folder_name_strings = []
    for seed in seeds_to_merge:
        if seed < 0:
            folder_name_strings.append("depth_5120")
        else:
            folder_name_strings.append(f"{exp}{seed}")
            
    # 2. Create the target output folder
    seeds_suffix = "_".join(map(str, seeds_to_merge))
    exp_output_folder_name = f"{exp}_{seeds_suffix}"
    exp_output_dir = os.path.join(OUTPUT_DIR, exp_output_folder_name)
    os.makedirs(exp_output_dir, exist_ok=True)
    print(f"Output directory for {exp}: {exp_output_dir}")
    
    # 3. Go through all PNG files in the first folder
    first_folder = folder_name_strings[0]
    first_folder_path = os.path.join(INPUT_BASE_DIR, first_folder)
    
    if not os.path.isdir(first_folder_path):
        print(f"Warning: First folder does not exist: {first_folder_path}")
        continue
        
    # Get all PNG files in the first folder
    png_files = glob.glob(os.path.join(first_folder_path, "*.png"))
    
    for png_file_path in png_files:
        png_filename = os.path.basename(png_file_path)
        
        images = []
        missing_count = 0
        missing_info = []
        
        for idx, folder_name in enumerate(folder_name_strings):
            seed_val = seeds_to_merge[idx]
            img_path = os.path.join(INPUT_BASE_DIR, folder_name, png_filename)
            
            if os.path.exists(img_path):
                try:
                    img = Image.open(img_path)
                    images.append(img)
                except Exception as e:
                    print(f"Error opening image {img_path}: {e}")
                    missing_count += 1
                    missing_info.append(f"seed {seed_val} (folder: {folder_name})")
            else:
                missing_count += 1
                missing_info.append(f"seed {seed_val} (folder: {folder_name})")
                
        if missing_count > 0:
            print(f"Warning for {png_filename}: {missing_count} images were missing. Missing for: {', '.join(missing_info)}")
            
        if not images:
            continue
            
        # Combine images vertically
        widths, heights = zip(*(i.size for i in images))
        
        max_width = max(widths)
        total_height = sum(heights)
        
        new_im = Image.new('RGB', (max_width, total_height), color='white')
        
        y_offset = 0
        for im in images:
            new_im.paste(im, (0, y_offset))
            y_offset += im.size[1]
            
        out_path = os.path.join(exp_output_dir, png_filename)
        new_im.save(out_path)
        print(f"Saved combined image to {out_path}")



--- Processing experiment: query_mask_5_Seed ---
Output directory for query_mask_5_Seed: C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds\query_mask_5_Seed_-1_0_1_2_3_7
Saved combined image to C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds\query_mask_5_Seed_-1_0_1_2_3_7\ASCT2.png
Warning for CCR5.png: 4 images were missing. Missing for: seed 0 (folder: query_mask_5_Seed0), seed 1 (folder: query_mask_5_Seed1), seed 2 (folder: query_mask_5_Seed2), seed 3 (folder: query_mask_5_Seed3)
Saved combined image to C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds\query_mask_5_Seed_-1_0_1_2_3_7\CCR5.png
Saved combined image to C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds\query_mask_5_Seed_-1_0_1_2_3_

# RENAMING log.txt files

In [ ]:
"""
Documentation:
This script renames 'log.txt' files found within the immediate subdirectories (depth 1) of a specified input path.

1. It looks inside `RENAMING_INPUT_PATH`.
2. It iterates through all immediate folders (it does not search deeper inside those subfolders).
3. If it finds a file named `log.txt` inside a folder, it extracts a suffix from the folder's name. The suffix is everything from the last underscore ("_") to the end of the folder name.
4. It then renames `log.txt` by appending this suffix to "log".
   (e.g., in folder "qu_mask_15_ASCT2_AlaRepSEEDchange0", the extracted suffix is "_AlaRepSEEDchange0", and the file is renamed to "log_AlaRepSEEDchange0.txt").
"""

import os

# The specifiable Windows path (set to your default)
# Using 'r' before the string ensures Windows backslashes are treated as literal characters.
RENAMING_INPUT_PATH = r"C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\RENAMING"


def rename_logs_in_folders(base_path):
    # Check if the specified directory exists
    if not os.path.exists(base_path):
        print(f"Error: The path '{base_path}' does not exist.")
        return

    # Iterate through all items in the base directory
    for item_name in os.listdir(base_path):
        folder_path = os.path.join(base_path, item_name)
        
        # Check if the item is a directory (we only want depth 1 folders)
        if os.path.isdir(folder_path):
            log_file_path = os.path.join(folder_path, "log.txt")
            
            # Check if a file named 'log.txt' exists in this folder
            if os.path.exists(log_file_path) and os.path.isfile(log_file_path):
                
                # Find the index of the last underscore in the folder's name
                last_underscore_index = item_name.rfind('_')
                
                if last_underscore_index != -1:
                    # Extract the suffix starting from the last underscore (e.g., "_AlaRepSEEDchange0")
                    suffix_with_underscore = item_name[last_underscore_index:]
                    
                    # Create the new filename.
                    # 'suffix_with_underscore' already includes the '_' so we just append it directly to 'log'
                    new_filename = f"log{suffix_with_underscore}.txt"
                    new_file_path = os.path.join(folder_path, new_filename)
                    
                    # Rename the file
                    try:
                        os.rename(log_file_path, new_file_path)
                        print(f"Renamed: '{log_file_path}'\n      -> '{new_file_path}'\n")
                    except Exception as e:
                        print(f"Failed to rename '{log_file_path}': {e}")
                else:
                    print(f"Skipped folder '{item_name}': No underscore found in the folder name.")

if __name__ == "__main__":
    rename_logs_in_folders(RENAMING_INPUT_PATH)
    print("Done processing folders.")


Renamed: 'C:\Users\franc\Downloads\testName\qu_mask_15_ASCT2_Seed0\log.txt'
      -> 'C:\Users\franc\Downloads\testName\qu_mask_15_ASCT2_Seed0\log_Seed0.txt'

Done processing folders.
